In [2]:
"""
PIPELINE STAGE: Smart Temporal Consolidation & Automated Schema Normalization
INPUTS: 03_2020_2021_final.xlsx, 06_2021_2024_final.xlsx
OUTPUT: 07_final_temporal_consolidation.xlsx
LIBRARIES: pandas, os

1. OBJECTIVE:
    Intelligently merge two longitudinal datasets with potentially mismatched headers into a 
    single, continuous time-series database. This script ensures structural alignment by 
    using the first dataset as a master schema reference.

2. SMART SCHEMA NORMALIZATION (Critical Logic):
    A. Header Synchronization: To prevent data fragmentation caused by different naming 
       conventions, the script treats the first file as the "Master Table".
    B. Automated Column Mapping: If column counts match, the script automatically overwrites 
       the second file's headers with the master headers, ensuring a seamless vertical 
       concatenation without data shifting.
    C. Validation Check: Implements a safeguard mechanism to warn the user if column counts 
       are inconsistent, preventing structural corruption.

3. DATA INTEGRATION & CONCATENATION:
    - Method: Vertical concatenation (appending rows) across temporal boundaries.
    - Index Management: Reset indices (ignore_index=True) to create a clean, continuous 
      primary key sequence for the unified dataset.

4. AUDIT & REPORTING:
    - Real-time logging of the reading process, individual row counts per file, and the 
      final consolidated record count.
    - Transparency: Detailed terminal output to verify successful data ingestion.

5. ERROR RESILIENCE:
    - Comprehensive try-except encapsulation to capture I/O exceptions or memory errors.
    - Descriptive error reporting for user-friendly debugging.
"""


import pandas as pd
import os

def excel_birlestirme_operasyonu():
    # 1. Klasör ve dosya yollarını tanımla
    dosya1_yol = os.path.join("..", "..", "processed_data", "steps", "03_2020_2021_final.xlsx")
    dosya2_yol = os.path.join("..","..", "processed_data", "steps", "06_2021_2024_final.xlsx")
    cikti_yol = os.path.join("..", "..", "processed_data", "steps", "07_final_energy_consolidation.xlsx")

    try:
        # 2. İlk dosyayı oku (Başlıkları buradan alacağız)
        df1 = pd.read_excel(dosya1_yol)
        print(f"İlk dosya okundu: {dosya1_yol} ({len(df1)} satır)")

        # 3. İkinci dosyayı oku
        # header=0 diyerek mevcut başlıkları okuyoruz ama sonra ezeceğiz
        df2 = pd.read_excel(dosya2_yol)
        
        # İkinci dosyanın sütun isimlerini, ilk dosyanın isimleriyle birebir değiştir
        # Bu işlem başlıkları değil, sadece altındaki veriyi df1 formatına sokar
        if len(df1.columns) == len(df2.columns):
            df2.columns = df1.columns
            print(f"İkinci dosya başlıkları normalize edildi: {dosya2_yol} ({len(df2)} satır)")
        else:
            print("UYARI: Dosyaların sütun sayıları eşit değil! Veri kayması olabilir.")

        # 4. İki tabloyu alt alta birleştir
        # ignore_index=True ile satır numaralarını 0'dan başlayacak şekilde yeniden düzenle
        final_df = pd.concat([df1, df2], ignore_index=True)

        # 5. Sonucu yeni dosya olarak kaydet
        final_df.to_excel(cikti_yol, index=False)
        
        print("-" * 30)
        print(f"İŞLEM BAŞARILI!")
        print(f"Toplam satır sayısı: {len(final_df)}")
        print(f"Kayıt yeri: {cikti_yol}")

    except Exception as e:
        print(f"HATA OLUŞTU: {e}")

if __name__ == "__main__":
    excel_birlestirme_operasyonu()

İlk dosya okundu: ..\..\processed_data\steps\03_2020_2021_final.xlsx (1782 satır)
İkinci dosya başlıkları normalize edildi: ..\..\processed_data\steps\06_2021_2024_final.xlsx (3078 satır)
------------------------------
İŞLEM BAŞARILI!
Toplam satır sayısı: 4860
Kayıt yeri: ..\..\processed_data\steps\07_final_energy_consolidation.xlsx
